# Neighborhoods with Most Tree Canopy Change

Which neighborhoods lose tree canopy the fastest?

Are there hotspots of tree canopy loss or gain?

In [55]:
import rasterio
import geopandas as gpd
import pandas as pd
import numpy as np
import re
from rasterio.mask import mask
from shapely.geometry import mapping

In [56]:
lcdf = pd.read_csv('lulc_visualization_crosswalk-2024-edition.csv')

In [57]:
#change rasters found at: https://www.sciencebase.gov/catalog/item/68011c83d4be0263cab101c8

change1to2 = "../landcover_spring2026_toobig/lcc/balt_24510_lulc-change_2013-2018_2024-Edition.tif"
change1to3 = "../landcover_spring2026_toobig/lcc/balt_24510_lulc-change_2013-2021_2024-Edition.tif"                      
change2to3 = "../landcover_spring2026_toobig/lcc/balt_24510_lulc-change_2018-2021_2024-Edition.tif"


In [58]:
#land cover raster

balt_2013 = "../landcover_spring2026_toobig/lc/balt_24510_lulc_2013_2024-Edition.tif"
balt_2018 = "../landcover_spring2026_toobig/lc/balt_24510_lulc_2018_2024-Edition.tif"
balt_2021 = "../landcover_spring2026_toobig/lc/balt_24510_lulc_2021_2024-Edition.tif"


In [5]:
re.findall(r'(\d{4})',balt_2013)[2]

'2013'

In [59]:
def landcover_polygon(
    raster_path,
    lookup_df,
    polygons_gdf,
    #output_csv,
    pixel_col="pixel_value",
    class_col="class_name",
    neighborhood_name_col="Name"
):

    #use geodataframe with neighborhood_id
    gdf = polygons_gdf.copy()

    if "neighborhood_id" not in gdf.columns:
        raise ValueError("GeoDataFrame must contain 'neighborhood_id'")

    # Build pixel value → class dictionary
    lookup_df = lookup_df.copy()

    

    value_to_class = dict(zip(lookup_df[pixel_col], lookup_df[class_col]))

    results = []

    with rasterio.open(raster_path) as src:

        # Ensure CRS match
        if gdf.crs != src.crs:
            gdf = gdf.to_crs(src.crs)

        for idx, row in gdf.iterrows():

            geom = [mapping(row.geometry)]

            out_image, _ = mask(src, geom, crop=True)
            data = out_image[0]
            # mask(src, geom, crop=True)[0]

            # Remove nodata
            data = data[data != src.nodata]

            if len(data) == 0:
                continue

            # Convert pixel values to class names
            classes = pd.Series(data).map(value_to_class)

            # Count pixels per class
            class_counts = classes.value_counts()

            # Convert to acres
            
            class_acres = class_counts / 4046.8564224

            total_acres = class_acres.sum()

            row_dict = {
                "neighborhood_id": row["neighborhood_id"],
                neighborhood_name_col: row[neighborhood_name_col]
            }

            #print(row["neighborhood_id"])
            #print(row[neighborhood_name_col])

            total_percent = 0

            for lc_class, acres in class_acres.items():
                
                percent = (acres / total_acres) * 100 if total_acres > 0 else 0
    
                row_dict[f"{lc_class}_acres".replace(" ","_")] = acres
                row_dict[f"{lc_class}_percent".replace(" ","_")] = percent
    
                total_percent += percent

            results.append(row_dict)

    results_df = pd.DataFrame(results).fillna(0)
    results_df["total_percent"] = total_percent
    final_df = results_df.copy()


    return final_df


In [60]:
#lookup table function


lcdf = pd.read_csv('lulc_visualization_crosswalk-2024-edition.csv') 


def merge_classes(lookup_df, class_col="class_name"):
    
    lookup_df = lookup_df.copy()
    
    lookup_df[class_col] = lookup_df[class_col].str.lower()
    
    def merge_name(name):
        if "tree canopy" in name:
            return "tree canopy"
        elif "impervious" in name:
            return "impervious"
        else:
            return name
    
    lookup_df[class_col] = lookup_df[class_col].apply(merge_name)
    
    return lookup_df

crosswalk_lc = merge_classes(lcdf, class_col="lc") 
pd.reset_option('display.max_rows')
crosswalk_lc

,value,lulc,lu,lc,general,macro
0,10,Tidal Waters,Tidal Waters,water,Water,Water
1,11,Lakes and Reservoirs,Lakes and Reservoirs,water,Water,Water
2,12,Riverine Ponds,Riverine Ponds,water,Water,Water
3,13,Terrene Ponds,Terrene Ponds,water,Water,Water
4,14,Streams and Rivers,Streams and Rivers,water,Water,Water
...,...,...,...,...,...,...
57,91,Animal Operation Impervious,Animal Operation,impervious,"Impervious, Other",Agricultural
58,92,Animal Operation Barren,Animal Operation,barren,Pasture and Hay,Agricultural
59,93,Animal Operation Herbaceous,Animal Operation,low vegetation,Pasture and Hay,Agricultural
60,94,Tree Canopy over Agricultural Structure,Animal Operation,tree canopy,Tree Canopy over Impervious,Agricultural


In [61]:

#difference between years in strings 

import re

#regex to get years from column names
#int(re.search(r'(\d{4})',name).group(1))

int(re.search(r'(\d{4})','canopy_change_2018').group(1)) - int(re.search(r'(\d{4})','canopy_change_2013').group(1))


5

In [62]:
#combined function
#every row a neighborhood year pair

def lc_years(
    raster_paths, #input a list
    lookup_df,
    polygons_gdf,
    #output_csv,
    pixel_col="pixel_value",
    class_col="class_name",
    neighborhood_name_col="Name"
):

    #use geodataframe with neighborhood_id
    gdf = polygons_gdf.copy()

    if "neighborhood_id" not in gdf.columns:
        raise ValueError("GeoDataFrame must contain 'neighborhood_id'")

    # Build pixel value to class dictionary
    lookup_df = lookup_df.copy()


    value_to_class = dict(zip(lookup_df[pixel_col], lookup_df[class_col]))

    results = []

    for path in raster_paths:
        #get year from path with regex
        year = int(re.findall(r'(\d{4})',path)[2])

        with rasterio.open(path) as ras:

            # Ensure CRS match
            if gdf.crs != ras.crs:
                gdf = gdf.to_crs(ras.crs)

            for idx, row in gdf.iterrows():

                geom = [mapping(row.geometry)]

                out_image, _ = mask(ras, geom, crop=True)
                data = out_image[0]
                

                # Remove nodata
                data = data[data != ras.nodata]

                if len(data) == 0:
                    continue

                # Convert pixel values to class names
                classes = pd.Series(data).map(value_to_class).str.lower()


                # rename categories
                classes1 = classes.apply(lambda x: "tree_canopy" if "tree canopy" in x
                                       else x)

                classes2 = classes1.apply(lambda x: "impervious" if "impervious" in x
                                       else x)

                class_counts = classes2.value_counts()


                row_dict = {
                    "neighborhood_id": row["neighborhood_id"],
                    neighborhood_name_col: row[neighborhood_name_col]
                }

                row_dict['year'] = year

                #print(row["neighborhood_id"])
                #print(row[neighborhood_name_col])

                total_percent = 0

                #convert to acres

                #t_acres = class_counts.get("tree_canopy", 0) / 4046.8564224
                #i_acres = class_counts.get("impervious", 0) / 4046.8564224

                class_acres = class_counts / 4046.8564224

                total_acres = class_acres.sum()

                filtered = [(lc_class, acres) for lc_class, acres in class_acres.items() if lc_class in ["tree_canopy","impervious"]]

                for lc_class, acres in filtered:
                
                    percent = (acres / total_acres) * 100 if total_acres > 0 else 0
    
                    row_dict[f"{lc_class}_acres".replace(" ","_")] = acres
                    row_dict[f"{lc_class}_percent".replace(" ","_")] = percent
                

                for lc_class, acres in class_acres.items():
                    percent = (acres / total_acres) * 100 if total_acres > 0 else 0
                    total_percent += percent


                results.append(row_dict)

    results_df = pd.DataFrame(results).fillna(0)
    results_df["total_percent"] = total_percent
    final_df = results_df.copy()


    return final_df



In [63]:
#assign neighbordhood_id to gdf for future use

baltcn = "../landcover_spring2026/BES_neighborhoodDemographicChange1930_2010/BES_neighborhoodDemographicChange1930_2010.shp"
baltcn_gdf = gpd.read_file(baltcn)

gdf = baltcn_gdf.copy().reset_index(drop=True)
gdf["neighborhood_id"] = gdf.index + 1

In [64]:
lcdf = pd.read_csv('lulc_visualization_crosswalk-2024-edition.csv')
rasters = [balt_2013,balt_2018,balt_2021]

lc_years(
    rasters,
    lcdf,
    gdf,
    pixel_col="value",
    class_col="lc",
    neighborhood_name_col="Name"
)

,neighborhood_id,Name,year,impervious_acres,impervious_percent,tree_canopy_acres,tree_canopy_percent,total_percent
0,1,Abell,2013,32.860074,70.345645,9.189108,19.671706,100.0
1,2,Allendale,2013,82.710619,31.807686,51.130304,19.662973,100.0
2,3,Arcadia,2013,37.971696,26.245529,38.541768,26.639556,100.0
3,4,Arlington,2013,61.554445,53.255371,29.960292,25.920898,100.0
4,5,Armistead Gardens,2013,89.525785,29.585901,122.529922,40.492894,100.0
...,...,...,...,...,...,...,...,...
829,274,Downtown West,2021,64.811294,92.309263,3.633932,5.175727,100.0
830,275,Belair-Edison,2021,337.146876,63.854597,78.739388,14.913002,100.0
831,276,Four By Four,2021,26.167966,57.740604,13.470209,29.722524,100.0
832,277,Charles Village,2021,125.838168,74.658338,30.500959,18.095868,100.0


In [50]:

#annualization function

def annualization(
    raster_paths,
    lookup_df,
    polygons_gdf,
    pixel_col="pixel_value",
    class_col="class_name",
    neighborhood_name_col="Name"
):
    #create land cover years dataframe
    df = lc_years(
    raster_paths=raster_paths,
    lookup_df=lookup_df,
    polygons_gdf=polygons_gdf,
    pixel_col=pixel_col,
    class_col=class_col,
    neighborhood_name_col=neighborhood_name_col
    )
    
    df_sorted = df.sort_values(["neighborhood_id", "year"])

    results = []

    for (nei_id, name), group in df_sorted.groupby(["neighborhood_id", neighborhood_name_col]):

        earliest_year = group["year"].min()
        latest_year = group["year"].max()

        earliest_acres = group.loc[group["year"] == earliest_year, "tree_canopy_acres"].iloc[0]
        latest_acres = group.loc[group["year"] == latest_year, "tree_canopy_acres"].iloc[0]

        years_diff = latest_year - earliest_year

        annual_change = (latest_acres - earliest_acres) / years_diff if years_diff > 0 else 0

        results.append({
            "neighborhood_id": nei_id,
            neighborhood_name_col: name,
            "start_year": earliest_year,
            "end_year": latest_year,
            "tree_canopy_acres_start": earliest_acres,
            "tree_canopy_acres_end": latest_acres,
            "annual_tree_canopy_change_acres_per_year": annual_change
        })

    #
    return pd.DataFrame(results)

    

In [54]:
annualization(
    rasters,
    lcdf,
    gdf,
    pixel_col="value",
    class_col="lc",
    neighborhood_name_col="Name"
)


,neighborhood_id,Name,start_year,end_year,tree_canopy_acres_start,tree_canopy_acres_end,annual_tree_canopy_change_acres_per_year
0,1,Abell,2013,2021,9.189108,9.358128,0.021128
1,2,Allendale,2013,2021,51.130304,49.104534,-0.253221
2,3,Arcadia,2013,2021,38.541768,38.700656,0.019861
3,4,Arlington,2013,2021,29.960292,29.335857,-0.078054
4,5,Armistead Gardens,2013,2021,122.529922,125.113655,0.322967
...,...,...,...,...,...,...,...
273,274,Downtown West,2013,2021,3.982845,3.633932,-0.043614
274,275,Belair-Edison,2013,2021,76.305400,78.739388,0.304249
275,276,Four By Four,2013,2021,13.478116,13.470209,-0.000988
276,277,Charles Village,2013,2021,30.219258,30.500959,0.035213
